# Open Problems & Research Frontier

This is the final notebook in the series, and it's about what we **don't** know yet. We'll survey the key open problems in mechanistic interpretability as of early 2025, drawing heavily from [Open Problems in Mechanistic Interpretability](https://arxiv.org/abs/2501.16496) (Bereska & Gavves, 2025).

Mostly markdown with some illustrative code -- think of it as your map to the frontier.

## 1. The Validation Problem

### "Explanation Theater"

This is the biggest methodological challenge in interpretability, and you should think about it every time you look at a result.

**The problem:** Interpretability methods produce outputs (feature labels, circuits, attribution graphs) that *look* interpretable. But how do you know they're correct?

### Current validation approaches

- **Ablation tests**: Remove the identified component -- does behavior change as predicted?
- **Counterfactual testing**: If the explanation is correct, what else should be true?
- **Causal scrubbing** (Redwood Research): Formal framework for testing whether a computational graph fully explains behavior
- **Prediction**: Can the interpretation predict model behavior on new inputs?

### Open questions

- What counts as sufficient evidence that an interpretation is correct?
- How do we avoid confirmation bias when interpreting features?
- Can we automate validation (auto-interp evaluation)?

## 2. The Scalability Problem

Most mechanistic interpretability has been done on small models (GPT-2, Pythia). Scaling to frontier models hits several walls:

1. **Computational cost**: Activation patching on GPT-4-scale models is extremely expensive
2. **Feature explosion**: SAEs on larger models find millions of features -- how do you navigate that?
3. **Circuit complexity**: Circuits in larger models may be qualitatively different (more distributed, more redundant)
4. **Tooling**: Current tools struggle with models above ~7B parameters

### Progress

- Anthropic's [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/) (2024) showed SAEs work on Claude 3 Sonnet
- [Circuit tracing](https://transformer-circuits.pub/2025/attribution-graphs/methods.html) (2025) was demonstrated on Claude 3.5 Haiku
- Gemma Scope provides SAEs for 2B and 9B parameter models
- NNsight enables remote access to larger models

**Key question:** Do the interpretable structures found in small models exist in the same form at scale? Or does something qualitatively change?

## 3. The Features-to-Circuits Gap

We can extract features (via SAEs) and we can find circuits (via patching). But connecting them into a full understanding is still hard.

### The pipeline we want

```
Raw activations -> Features (SAEs) -> Circuits (patching/tracing) -> Algorithms (understanding) -> Behavior (prediction)
```

### Where we actually are

- **Features -> Circuits**: Circuit tracing (2025) makes real progress here
- **Circuits -> Algorithms**: Still largely manual / requires human insight
- **Algorithms -> Behavior**: Mostly limited to simple tasks (IOI, greater-than)

### Open problems

- Can we automatically identify the "algorithm" a circuit implements?
- How do we handle distributed computation (where no single circuit is responsible)?
- How do features compose? What's the algebra of features?

## 4. Superposition — Unsolved Aspects

Superposition (see notebook 02) is well-described theoretically, but many questions remain:

1. **Are SAEs finding the "right" features?** The decomposition isn't unique — different SAEs on the same activations find different features. Which is correct?

2. **Feature splitting and absorption**: As SAE width increases, features split into finer-grained sub-features. When do we stop? Is there a "natural" granularity?

3. **Beyond linear features**: What if some features are non-linear directions (curves in activation space)? SAEs can only find linear features.

4. **Multi-token features**: Some concepts span multiple token positions. Current SAEs operate position-by-position.

5. **Computation in superposition**: We understand how features are *stored* in superposition. How is *computation* performed on superposed representations?

The following cell illustrates the concept of **feature splitting** — how features become progressively finer-grained as SAE width increases.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Illustrate the concept of feature splitting
# As SAE width increases, broad features split into specific sub-features

# Simulated example
widths = [256, 1024, 4096, 16384, 65536]
descriptions = {
    256: ["animal", "vehicle", "food", "place"],
    1024: ["dog", "cat", "bird", "car", "truck", "fruit", "meat", "city", "country"],
    4096: ["golden retriever", "tabby cat", "robin", "sparrow", "sedan", "pickup truck",
           "apple", "banana", "steak", "chicken", "New York", "London", "France", "Japan"],
    16384: ["golden retriever puppy", "adult golden retriever", "orange tabby", "gray tabby",
            "American robin", "European robin", "house sparrow", "tree sparrow", "..."],
    65536: ["golden retriever puppy playing", "golden retriever puppy sleeping",
            "adult golden retriever outdoors", "adult golden retriever indoors", "..."],
}

fig, ax = plt.subplots(figsize=(12, 6))
for i, (w, descs) in enumerate(descriptions.items()):
    n = min(len(descs), 6)
    for j, d in enumerate(descs[:n]):
        ax.text(i, j, d, fontsize=8, ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=f'C{j%10}', alpha=0.3))

ax.set_xticks(range(len(widths)))
ax.set_xticklabels([f"{w:,}" for w in widths])
ax.set_xlabel("SAE Width (number of features)")
ax.set_ylabel("Feature Examples")
ax.set_title("Feature Splitting: Broader → Finer Features as SAE Width Increases")
ax.set_ylim(-1, 7)
ax.set_xlim(-0.5, len(widths) - 0.5)
plt.tight_layout()
plt.show()

## 5. Safety Applications -- The Ultimate Goal

The whole reason we do interpretability is **AI safety**. Here are the key open questions:

### Can we detect deception?

If a model is "deceiving" its user, can interpretability catch it? Anthropic's circuit tracing found deception-related features, but:
- Can we monitor these in real-time?
- Can a deceptive model learn to hide its deception from our tools?
- What does the arms race dynamic look like?

### Can we verify alignment?

The ambitious goal: use interpretability to verify that a model's internal reasoning is aligned with human values. This requires:
- Understanding what "aligned reasoning" looks like in activation space
- Comprehensive coverage (not just spot-checking specific behaviors)
- Robustness to distribution shift

### Can we improve models?

Beyond monitoring, can interpretability help us build better models?
- Remove harmful capabilities without retraining
- Enhance specific capabilities by amplifying relevant circuits
- Debug failures by tracing the computational path that led to an error

### The meta-question

When we use LLMs to interpret other LLMs (auto-interp), how do we validate the interpreter? We need interpretability tools that don't themselves require interpretation.

## 6. Research Directions for 2025-2026

Based on where the field is heading:

1. **Automated mechanistic interpretability**: LLM-powered pipelines that automatically discover, label, and validate features and circuits. Less reliance on human researchers.

2. **Real-time monitoring**: Using interpretability features (from SAEs) as a monitoring signal during deployment. Flag when concerning features activate.

3. **Interpretability-aware training**: Training models that are *easier* to interpret from the start. Concept bottleneck architectures, modular designs.

4. **Cross-model comparison**: Comparing internal representations of different models. Do GPT-4 and Claude share the same features? Are there universal features?

5. **Formal verification via interpretability**: Using mechanistic understanding to provide formal guarantees about model behavior (distant goal but actively researched).

6. **Multimodal interpretability**: Extending these techniques to vision-language models, code models, and other modalities. Prisma (2025) is an early step.

7. **Developmental interpretability**: Understanding how features and circuits *form during training*. When do induction heads emerge? How do SAE features change over the course of training?

---
### Running Example: IOI — What We Still Don't Understand

This is the final installment of our **running example** investigating how GPT-2-small handles the Indirect Object Identification (IOI) task across all techniques in this guide.

**The task**: "When Mary and John went to the store, John gave a drink to" — the model should predict "Mary".

The IOI circuit is one of the most thoroughly studied circuits in mechanistic interpretability. Yet even for this well-understood task, significant open questions remain:

- **Backup name mover heads**: The IOI circuit contains redundant "backup" name mover heads that activate when primary name movers are ablated. We don't fully understand why the model develops this redundancy during training, or whether it serves a purpose beyond robustness.

- **Predictability from architecture**: We cannot predict from the transformer architecture alone that this particular circuit would form. Given the same architecture and a different random seed, does the same circuit emerge? (Early evidence suggests: similar but not identical.)

- **Discovery was manual**: The IOI circuit was found through painstaking human effort (Wang et al., 2022). Fully automated discovery of equally complex circuits — without human guidance on what to look for — remains an open problem. ACDC and circuit tracing are steps toward this, but are not yet fully autonomous.

- **Scale dependence**: We don't know if larger models (GPT-3, GPT-4, Claude) use the same circuit structure for IOI. They might use fundamentally different algorithms, or the same algorithm distributed across more components. Cross-model circuit comparison is still in its infancy.

- **Completeness**: Even the "complete" IOI circuit explanation accounts for only ~85% of the logit difference. The remaining 15% is distributed across many small contributions. Is this residual important, or just noise?

These open questions for IOI are microcosms of the field's broader challenges. If we cannot fully understand a 26-head circuit in a 12-layer model, the path to understanding frontier model behavior is long — but the techniques covered in this guide are the best tools we have.

## Exercises

### Exercise 1: Design a Validation Experiment

Pick one interpretability claim from this guide (e.g., "head 9.9 is a name mover head in the IOI circuit" or "the logit lens shows predictions forming gradually across layers"). Design an experiment that would **falsify** this claim if it were wrong.

Your experiment should address:
1. What specific, testable prediction does the claim make?
2. What inputs or conditions would violate this prediction?
3. What result would constitute strong evidence against the claim?

Good falsification strategies include:
- Testing the claim on **out-of-distribution** inputs (does the head still move names in non-IOI contexts?)
- Checking if the component does **only** what is claimed, or has other functions too
- Testing **necessary vs sufficient** conditions (is the component necessary for the behavior, or just correlated?)

<details>
<summary>Hint</summary>

For example, to falsify "head 9.9 is a name mover head": (1) Test on prompts with 3+ names — does L9H9 still attend to the correct indirect object? (2) Test on prompts where the indirect object appears in an unusual position. (3) Zero-ablate L9H9 and check if other heads compensate (backup behavior) — if so, L9H9 may not be *necessary*. (4) Check if L9H9 does anything else — run it on non-IOI prompts and see what it attends to. A strong falsification would show that the head is not specialized for name-moving, or that the circuit works fine without it.

</details>

In [ ]:
# TODO: Try modifying this!
# Claim: "Head 9.9 is a name mover head that copies the indirect object name to the output"

from transformer_lens import HookedTransformer
import torch

model = HookedTransformer.from_pretrained("gpt2-small")

# Falsification test: does L9H9 still move the correct name with 3+ names?
test_prompts = [
    # Standard IOI (2 names)
    ("When Mary and John went to the store, John gave a drink to", " Mary"),
    # 3 names — does L9H9 still pick the right indirect object?
    ("When Mary and John and Alice went out, John gave a drink to", " Mary"),
    # Reversed order
    ("When John and Mary went to the store, Mary gave a drink to", " John"),
    # Different names entirely
    ("When Sarah and Bob went to the park, Bob gave a ball to", " Sarah"),
    # Non-IOI prompt — what does L9H9 attend to here?
    ("The capital of France is the city of", " Paris"),
]

print("Falsification test: Is L9H9 a specialized name mover head?\n")
for prompt, expected in test_prompts:
    logits, cache = model.run_with_cache(prompt)
    target_id = model.to_single_token(expected)
    tokens = [model.tokenizer.decode(t) for t in model.to_tokens(prompt)[0]]
    
    # Check attention pattern of L9H9
    attn = cache["blocks.9.attn.hook_pattern"][0, 9, -1]  # L9H9, from last position
    top_attn_pos = attn.argmax().item()
    top_attn_token = tokens[top_attn_pos]
    
    # Check logit contribution of L9H9
    head_out = cache["blocks.9.attn.hook_result"][0, -1, 9]  # (d_model,)
    direct_logit = (head_out @ model.W_U[:, target_id]).item()
    
    # Zero-ablate L9H9 and measure effect
    def zero_99(activation, hook):
        activation[:, :, 9, :] = 0
        return activation
    
    patched_logits = model.run_with_hooks(prompt, fwd_hooks=[("blocks.9.attn.hook_result", zero_99)])
    logit_drop = logits[0, -1, target_id].item() - patched_logits[0, -1, target_id].item()
    
    print(f"Prompt: '{prompt[:50]}...'")
    print(f"  Expected: '{expected}' | L9H9 attends most to: '{top_attn_token}' (pos {top_attn_pos})")
    print(f"  Direct logit contribution: {direct_logit:.3f} | Logit drop when ablated: {logit_drop:.3f}")
    print()

print("If L9H9 is a true name mover, it should attend to the indirect object")
print("in IOI prompts and show large logit drops when ablated for those cases.")
print("On non-IOI prompts, it may do something different — which would show")
print("it is not *exclusively* a name mover head.")

### Exercise 2: Write a Research Proposal

This is a writing exercise. Pick one open problem from this notebook (validation, scalability, the features-to-circuits gap, unsolved aspects of superposition, or safety applications). Write a 1-page research proposal addressing:

1. **Specific question**: What concrete question would you answer? (Not "solve interpretability" but something like "Do induction heads in GPT-2 use the same OV circuit structure as induction heads in Pythia?")
2. **Method**: What tools and techniques would you use? What data would you need?
3. **Success criteria**: What would a positive result look like? What would a negative result look like? Both should be informative.
4. **Key risks**: What could go wrong? What assumptions might not hold?

<details>
<summary>Hint</summary>

A good research proposal is narrow enough to be feasible but broad enough to be interesting. For example: "I would investigate whether the IOI circuit generalizes across languages by testing GPT-2 multilingual variants on translated IOI prompts. I would use activation patching to check if the same heads are important. Success = same circuit structure across languages. Failure = different circuits, which would suggest language-specific processing. Risk = multilingual GPT-2 may not handle IOI well in non-English languages, making the comparison invalid."

</details>

In [ ]:
# TODO: Try modifying this!
# Research proposal skeleton — fill in your own question and run the feasibility check

research_question = "Does the IOI circuit generalize across sentence structures?"
method = "Activation patching on GPT-2-small using varied IOI templates"
success_metric = "If the same heads show high attribution across 5+ different sentence templates"
key_risk = "The model might use different circuits for different syntactic structures"

print("=== Research Proposal Sketch ===")
print(f"Question: {research_question}")
print(f"Method: {method}")
print(f"Success: {success_metric}")
print(f"Risk: {key_risk}")

# Quick feasibility check: does the IOI circuit even work on a different template?
prompt_templates = [
    "When Mary and John went to the store, John gave a drink to",
    "After Mary met John at the park, John handed a gift to",
    "Mary and John were talking, then John passed the book to",
    "While Mary and John ate dinner, John offered dessert to",
    "Mary visited John at work, and John sent a letter to",
]

print("\nFeasibility check — logit diff (Mary - John) across templates:")
for prompt in prompt_templates:
    logits = model(prompt)
    mary_id = model.to_single_token(" Mary")
    john_id = model.to_single_token(" John")
    diff = (logits[0, -1, mary_id] - logits[0, -1, john_id]).item()
    print(f"  '{prompt[:45]}...': logit diff = {diff:.2f}")

## 7. Getting Involved

### Research Programs

- **MATS (ML Alignment Theory Scholars)**: 500+ researchers, mentored by top interpretability researchers
- **SERI MATS**: Summer program for alignment research
- **Apart Research**: Runs hackathons and collaborative research sprints

### Key Venues

- **Transformer Circuits Thread**: https://transformer-circuits.pub/
- **Alignment Forum**: https://www.alignmentforum.org/
- **LessWrong**: https://www.lesswrong.com/ (Mechanistic Interpretability tag)
- **arXiv**: cs.LG, cs.AI, cs.CL

### Open Source Contributions

- **TransformerLens**: always needs contributors
- **SAELens**: training infrastructure improvements
- **Neuronpedia**: feature annotations and tooling
- **Reproduce papers**: many papers lack public reproductions

### Key Paper: Start Here

If you read one paper to understand the open problems:

**"Open Problems in Mechanistic Interpretability"** (Bereska & Gavves, 2025) — https://arxiv.org/abs/2501.16496

This provides a comprehensive map of what's solved, what's partially solved, and what remains wide open.

---

*This guide will be updated as the field evolves. Run the research tracker to see the latest papers.*